# Slice 2 — notebook prototype

**Question we're answering:** *Given a Windows disk image suspected of compromise, what persistence mechanisms did the attacker install?* (Slice 2 scope — `base-wkstn-05` E01 as the test image.)

Cells run top-to-bottom. Each cell is one step of the decomposed pipeline — run them one at a time, inspect the printed output, re-run any cell in isolation.

Current cells:

| # | Cell | What it proves |
|---|---|---|
| C1 | Setup | `.env` loaded, OpenRouter + Langfuse clients built, constants printed |
| C2 | Schemas | Pydantic contracts for every phase — round-trip a sample `Findings` |
| C3 | MCP smoke test | Our MCP server spawns inside `sift`, exposes **4 tools** (`fsstat_e01`, `fls_list`, `icat_extract`, `regripper_run`), and the full chain `fsstat → fls_list → icat_extract(SOFTWARE) → regripper_run(run)` works end-to-end against the real E01 |
| C4 | Pipeline graph | LangGraph `StateGraph` with 4 stub nodes, rendered as Mermaid + PNG so we can SEE the pipeline before implementing it |
| C5 | EXTRACT | Real LLM call (`google/gemini-3.1-flash-lite-preview` via OpenRouter) with structured output validated against `Candidates`, Langfuse-traced, writes `out/candidates.json` |
| C6 | PLAN | Real LLM call (`anthropic/claude-sonnet-4.6`) produces a `ToolPlan` using all 4 tools, declares `icat_extract → regripper_run` dependencies, constrains plugins to the allowlist — writes `out/tool_plan.json` |

**Coming next:** C7 human checkpoint → C8 EXECUTE → C9 INTERPRET. Each time we implement a node, re-run C4 to see the graph with the real node attached.

## C1 — Setup

Loads environment variables, builds the OpenRouter (OpenAI-compatible) client, wires Langfuse (v4 reads `LANGFUSE_*` env vars automatically), and pins the two constants every downstream cell will use: the case ID and the E01 path.

> **Heads up:** `langfuse` v4 is an OTel rewrite. If you've used `@observe` from v2 before, the decorator is still here but the init story is env-based. `get_client()` returns the singleton.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from langfuse import get_client

load_dotenv()  # picks up any .env in /workspace; compose env vars take precedence anyway

# ---- Case + evidence ----
# Case selection. Swap these two constants to run the pipeline against a different
# image. Pre-existing artifacts in out/ should be archived to out/runs/<prior-case>/
# before switching, so tool_plan.APPROVED does not auto-approve the new plan.
CASE_ID = "srl-2018-wkstn-05"
E01_PATH = "/mnt/hackathon/base-wkstn-05-cdrive.E01"
QUESTION = "Given a Windows disk image suspected of compromise, what persistence mechanisms did the attacker install?"

# ---- Per-step model routing (SKILL.md Phase 5c) ----
MODELS = {
    "extract":   "google/gemini-3.1-flash-lite-preview",   # cheap — mechanical enumeration
    "plan":      "anthropic/claude-sonnet-4.6",            # quality — the verification gate
    "execute":   "google/gemini-3.1-flash-lite-preview",   # cheap — tool runner
    "interpret": "anthropic/claude-sonnet-4.6",            # quality — structured finding synthesis
}

# ---- OpenRouter (OpenAI-compatible) ----
openrouter = OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

# ---- Langfuse (v4 — env-based auto init) ----
langfuse = get_client()
lf_ok = langfuse.auth_check()

# ---- Summary ----
print(f"case_id           {CASE_ID}")
print(f"question          {QUESTION}")
print(f"E01 (inside sift) {E01_PATH}")
print()
print("models")
for phase, model in MODELS.items():
    print(f"  {phase:<10} {model}")
print()
print(f"OpenRouter base   {openrouter.base_url}")
print(f"Langfuse host     {os.environ.get('LANGFUSE_HOST', '(unset)')}")
print(f"Langfuse auth OK  {lf_ok}")


case_id           srl-2018-wkstn-05
question          Given a Windows disk image suspected of compromise, what persistence mechanisms did the attacker install?
E01 (inside sift) /mnt/hackathon/base-wkstn-05-cdrive.E01

models
  extract    google/gemini-3.1-flash-lite-preview
  plan       anthropic/claude-sonnet-4.6
  execute    google/gemini-3.1-flash-lite-preview
  interpret  anthropic/claude-sonnet-4.6

OpenRouter base   https://openrouter.ai/api/v1/
Langfuse host     https://us.cloud.langfuse.com
Langfuse auth OK  True


## C2 — Schemas (Pydantic contracts for every phase)

Every artifact this pipeline produces — `candidates.json`, `tool_plan.json`, `raw_results.jsonl`, `findings.json` — round-trips through one of these models. Defining them **before** the LLM calls is the point: invalid output from a step fails fast, and JSON schemas generated from these classes are what the models see in `response_format`.

Design notes baked in (SKILL.md Phase 4 — Prompt Hardening):
- `PersistenceCategory` includes `NOT_FOUND` — explicit absence beats hallucinated findings.
- `PlannedStep.confidence` is a 3-level enum with no default — forces calibrated scoring.
- `ToolPlan.expected_findings_range` is the over-extraction guard.
- `Evidence.tool_call_id` is the FK that links every finding back to a real line in `tool_calls.jsonl`.
- `Findings.plan_digest` is the sha256 of the approved plan — tamper-evident audit chain.

In [2]:
# Schemas extracted to pipeline/schemas.py (Slice 5 Step 1). See that module
# for the full Pydantic type definitions, ATT&CK mapping, and CRITIC schema.
#
# Imported below: Confidence, PersistenceCategory, Classification, RuleId,
# FailureCode, ATTACK_MAPPING, ATTACK_TACTIC_ID, ATTACK_TACTIC_NAME,
# ArtifactCandidate, Candidates, PlannedStep, ToolPlan, RawResult,
# Evidence, Finding, Findings, RuleFailure, CritiqueResult, CriticDisagreement.
from pipeline.schemas import *

# Notebook-prelude imports — downstream cells (C4 PipelineState, C6/C8/C9
# inline Pydantic models, round-trip smoke test below) reference these names
# directly out of the notebook namespace. Kept here so extraction is strictly
# additive, not a breaking change to cells that import-by-proximity.
from datetime import datetime, timezone
from pydantic import BaseModel, Field, model_validator

# ---- Round-trip smoke test — proves the JSON contract works end-to-end ----
sample = Findings(
    case_id=CASE_ID,
    question=QUESTION,
    findings=[
        Finding(
            category="registry_run_key",
            mechanism=r"HKCU\Software\Microsoft\Windows\CurrentVersion\Run\updater",
            value=r"C:\Users\public\updater.exe",
            confidence="high",
            classification="attacker_persistence",
            evidence=[Evidence(tool_call_id="tc-demo-1234", output_excerpt="updater -> ...")],
            notes="sample — not a real finding; ruled out DFIR tools (no known responder agent signature), vendor products (not McAfee/VMware path), Windows defaults (not a stock Microsoft name)",
        )
    ],
    plan_digest="sha256:demo",
    started_at=datetime.now(timezone.utc),
    finished_at=datetime.now(timezone.utc),
)

# Serialize, re-parse, confirm equal — proves the round-trip contract
dumped = sample.model_dump_json(indent=2)
reparsed = Findings.model_validate_json(dumped)
print("round-trip OK:", reparsed == sample)
print()
print(dumped)


round-trip OK: True

{
  "case_id": "srl-2018-wkstn-05",
  "question": "Given a Windows disk image suspected of compromise, what persistence mechanisms did the attacker install?",
  "findings": [
    {
      "category": "registry_run_key",
      "mechanism": "HKCU\\Software\\Microsoft\\Windows\\CurrentVersion\\Run\\updater",
      "value": "C:\\Users\\public\\updater.exe",
      "confidence": "high",
      "classification": "attacker_persistence",
      "evidence": [
        {
          "tool_call_id": "tc-demo-1234",
          "output_excerpt": "updater -> ..."
        }
      ],
      "notes": "sample — not a real finding; ruled out DFIR tools (no known responder agent signature), vendor products (not McAfee/VMware path), Windows defaults (not a stock Microsoft name)",
      "attack_id": "T1547.001",
      "attack_name": "Registry Run Keys / Startup Folder",
      "attack_tactic_id": "TA0003",
      "attack_tactic_name": "Persistence"
    }
  ],
  "plan_digest": "sha256:demo",
  "sta

## C3 — MCP smoke test

Spawns our MCP server inside the `sift` container via `docker exec -i sift python3 /opt/mcp/server.py`, negotiates the MCP handshake over stdio, lists the exposed tools, and drives all four of them end-to-end against the real E01:

1. `fsstat_e01` — filesystem metadata
2. `fls_list` — root directory listing
3. `icat_extract` — pull the `SOFTWARE` hive bytes out of the image
4. `regripper_run` — parse Run keys out of the extracted hive

`SOFTWARE_INODE` is hard-coded — we looked it up once via `fls -r -p | grep config/SOFTWARE` for this specific E01. A real PLAN run (C6) discovers hive inodes dynamically via `fls_list` steps; we shortcut here so C3 is a proper end-to-end smoke test of the MCP plumbing and nothing else.

**What to look for:**
- Four tools listed: `fsstat_e01`, `fls_list`, `icat_extract`, `regripper_run`
- All four calls return `exit_code: 0`
- `icat_extract` result shows ~80 MB of bytes written under `<case>/analysis/extracted/SOFTWARE`
- `regripper_run` excerpt contains real `Microsoft\Windows\CurrentVersion\Run` entries (VMware, McAfee, etc. are the legit apps on this host; malicious persistence on this image likely lives in per-user `NTUSER.DAT` hives or in the `System` hive's Services key, both of which the PLAN (C6) can reach with the same four tools)

**If C3 fails:**
- `ModuleNotFoundError: No module named 'mcp'` → rebuild the sift image (`docker compose build sift && docker compose up -d sift`) so [../../docker/sift/Dockerfile](../../docker/sift/Dockerfile) can bake in `mcp` + `pydantic`
- `syntax error at /usr/local/bin/rip.pl line 75` → same rebuild — the Dockerfile patches an upstream bug in `rip.pl`

In [ ]:
import json as _json
import os
import uuid

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
from langfuse import propagate_attributes

from pipeline.schemas import PlannedStep, ToolPlan
from pipeline.mcp.tokens import compute_plan_digest, issue_token

# Slice 5 Step 0.5: streamable-HTTP transport over the internal Docker
# bridge. `sift-mcp` is the long-lived FastMCP endpoint; `MCP_TRANSPORT_TOKEN`
# is the shared bearer pinned in .env / docker-compose.yaml.
MCP_URL = "http://sift-mcp:8000/mcp"
MCP_HEADERS = {"Authorization": f"Bearer {os.environ['MCP_TRANSPORT_TOKEN']}"}

# Known-good inode for /Windows/System32/config/SOFTWARE on THIS specific E01 —
# obtained once via `fls -r -p <E01> | grep Windows/System32/config/SOFTWARE$`.
# A real PLAN run (C6+) discovers this dynamically via fls_list; C3 just shortcuts
# the lookup so the smoke test stays self-contained.
SOFTWARE_INODE = 47479

# Slice 5 Step 4/7: every MCP call now carries a capability token + plan_digest.
# For the smoke test, synthesize a minimal ToolPlan covering the 4 tools we'll
# call, then issue a short-TTL token against it. allowed_paths covers the E01
# evidence root AND the per-case extracted/ directory (regripper's hive_path).
_smoke_plan = ToolPlan(
    question="MCP smoke test",
    steps=[
        PlannedStep(step_id=1, tool="fsstat_e01",    args={}, purpose="smoke", depends_on=[], confidence="high"),
        PlannedStep(step_id=2, tool="fls_list",      args={}, purpose="smoke", depends_on=[], confidence="high"),
        PlannedStep(step_id=3, tool="icat_extract",  args={}, purpose="smoke", depends_on=[], confidence="high"),
        PlannedStep(step_id=4, tool="regripper_run", args={}, purpose="smoke", depends_on=[], confidence="high"),
    ],
    expected_findings_range=(0, 0),
)
_smoke_plan_digest = compute_plan_digest(_smoke_plan)
_smoke_token = issue_token(
    _smoke_plan,
    case_id=CASE_ID,
    allowed_paths=(
        "/mnt/hackathon/",
        f"/home/sansforensics/cases/{CASE_ID}/analysis/extracted/",
    ),
    ttl_seconds=600,
)
_smoke_token_json = _smoke_token.model_dump_json()


def _unwrap(result):
    """FastMCP wraps Pydantic return values as structured dicts — sometimes nested
    under a `result` key. Normalize to a plain dict so the rest of the cell is boring.
    """
    if result.isError:
        msgs = [getattr(c, "text", str(c)) for c in result.content]
        raise RuntimeError("\n".join(msgs))
    data = getattr(result, "structuredContent", None)
    if data is None and result.content:
        data = _json.loads(getattr(result.content[0], "text", "{}"))
    data = data or {}
    if set(data.keys()) == {"result"}:
        data = data["result"]
    return data


# C3 gets its own Langfuse session (smoke-<hex>) so infra-verification calls sort
# separately from real pipeline runs (srl-…-<hex>) and don't pollute cost rollups.
# Tag as phase:smoke for UI filters.
smoke_run_id = f"smoke-{uuid.uuid4().hex[:8]}"
print(f"[smoke_run_id] {smoke_run_id}")
print(f"[smoke_token ] token_id={_smoke_token.token_id[:8]}…  "
      f"allowed_tools={sorted(_smoke_token.allowed_tools)}")
print(f"[plan_digest ] {_smoke_plan_digest[:16]}…")
print()

with propagate_attributes(
    session_id=smoke_run_id,
    user_id=CASE_ID,
    tags=["phase:smoke"],
    metadata={"phase": "smoke"},
):
    # Outer span groups the four tool calls into one tree per smoke run. Without
    # this, each _call() would create an orphan trace under the session.
    with langfuse.start_as_current_observation(name="mcp_smoke_test", as_type="span") as smoke_span:
        async with streamablehttp_client(MCP_URL, headers=MCP_HEADERS) as (read, write, _get_session_id):
            async with ClientSession(read, write) as session:
                init = await session.initialize()
                tools = (await session.list_tools()).tools
                print(f"server         {init.serverInfo.name} v{init.serverInfo.version}")
                print(f"tools ({len(tools)}): {', '.join(sorted(t.name for t in tools))}")
                print()

                async def _call(name, args):
                    # Thread capability_token + plan_digest on every call — the
                    # server enforces them both (signature, expiry, case_id,
                    # tool scope, path scope, plan_digest binding).
                    call_args = {
                        **args,
                        "capability_token": _smoke_token_json,
                        "plan_digest": _smoke_plan_digest,
                    }
                    # One Langfuse "tool" span per MCP call. Inputs / outputs are
                    # first-class on the span so runs diff cleanly in the UI. Non-ok
                    # status → span flagged ERROR, trivial to filter in the session list.
                    with langfuse.start_as_current_observation(
                        name=name, as_type="tool", input=args,
                    ) as span:
                        ev = _unwrap(await session.call_tool(name, call_args))
                        status = ev["tool_execution_status"]
                        span.update(
                            output={
                                "tool_execution_status": status,
                                "raw_sha256": ev["raw_sha256"],
                                "n_injection_flags": len(ev.get("injection_flags", [])),
                                "expected_paths_covered": ev.get("expected_paths_covered", []),
                            },
                            metadata={"tool_call_id": ev["tool_call_id"], "token_id": ev["token_id"]},
                        )
                        if status != "ok":
                            span.update(level="ERROR", status_message=f"status={status}")
                        n_flags = len(ev.get("injection_flags", []))
                        flags_note = f"  flags={n_flags}" if n_flags else ""
                        print(f"  {name:<22}  status={status:<12}  raw_sha256={ev['raw_sha256'][:16]}…{flags_note}")
                        return ev

                r_fsstat = await _call("fsstat_e01",    {"case_id": CASE_ID, "e01_path": E01_PATH})
                r_fls    = await _call("fls_list",      {"case_id": CASE_ID, "e01_path": E01_PATH, "parent_inode": None, "recurse": False})
                r_icat   = await _call("icat_extract",  {"case_id": CASE_ID, "e01_path": E01_PATH, "inode": SOFTWARE_INODE, "dest_filename": "SOFTWARE"})
                # regripper's hive_path comes from icat's structured_fields.dest_path —
                # NOT the raw_path (which points at the MCP server's persistence layer,
                # not the orchestrator-visible hive location).
                r_rip    = await _call("regripper_run", {"case_id": CASE_ID, "hive_path": r_icat["structured_fields"]["dest_path"], "plugin": "run"})

        smoke_span.update(output={
            "n_tools": len(tools),
            "tools": sorted(t.name for t in tools),
            "all_ok": all(r["tool_execution_status"] == "ok" for r in (r_fsstat, r_fls, r_icat, r_rip)),
        })

langfuse.flush()

print()
print("=== fsstat_e01 — structured_fields (channel B; the agent surface) ===")
sf_fs = r_fsstat["structured_fields"]
print(f"  fs_type        = {sf_fs.get('fs_type')!r}")
print(f"  block_size     = {sf_fs.get('block_size')!r}")
print(f"  mft_offset     = {sf_fs.get('mft_offset')!r}")
print(f"  partition_count= {sf_fs.get('partition_count')!r}")
print()
print("=== icat_extract — structured_fields ===")
sf_ic = r_icat["structured_fields"]
print(f"  dest_path     = {sf_ic.get('dest_path')!r}")
print(f"  bytes_written = {sf_ic.get('bytes_written'):,}")
print(f"  sha256        = {sf_ic.get('sha256','')[:16]}…")
print(f"  magic_bytes   = {sf_ic.get('magic_bytes')!r}")
print()
print("=== regripper_run(plugin=run) — first 5 structured entries ===")
entries = r_rip["structured_fields"].get("entries", [])
print(f"  {len(entries)} entries total; showing up to 5:")
for e in entries[:5]:
    name = e.get("value_name", "")
    data = (e.get("value_data_safe", "") or "")[:80]
    print(f"    {name:<30}  {data}")

## C4 — Pipeline graph (LangGraph)

Defines the Slice 2 pipeline as a LangGraph `StateGraph` so we can **see the pipeline** before we implement it. Nodes are stubs right now — they print a label and return an empty state delta. We will replace them one at a time in later cells, then re-run this cell to watch the graph grow.

**Why LangGraph here** (instead of just calling functions in order):
- Free visualization — Mermaid source + a rendered PNG inline, every time you run the cell.
- Explicit state contract — every node reads/writes `PipelineState`; no hidden kwargs threading.
- Slice 3's self-correction loop slots in cleanly as an `add_conditional_edges()` call on the `critic` node.

**Reading the graph (Slice 3 Phase B topology, 2026-04-19):**

```
__start__ → extract → plan → execute → interpret → critic ─┬─ commit → __end__
                                       ↑          ↑        ├─ re_interpret → interpret
                                       │          │        ├─ re_plan → plan
                                       │          │        └─ escalate → human_review → __end__
                                       └──────────┴──── (retry loop)
```

The Critic runs 11 deterministic rules (C10) on every finding INTERPRET produced; `critic_edge` (C12) inspects state and picks one of four branches. On `re_interpret` / `re_plan`, `corrective_instruction` is filled on state for the upstream stage to consume on retry (C6/C9 amendment is the next surgery — for now the prompts do not yet read it).

The human checkpoint between `plan` and `execute` lives outside LangGraph for now (a marker file the notebook asserts on). We may fold it in as an explicit `checkpoint` node later.

> If this cell fails with `ModuleNotFoundError: No module named '`langgraph`'`, the package was added to `pyproject.toml` but not yet installed in the running venv. Fix: from a terminal on the host, `docker exec find-evil-notebook uv sync` (or restart the notebook container, which re-runs `uv sync` on boot).

In [ ]:
# Slice 5 Step 7: PipelineState + graph build live in pipeline/graph.py.
# This cell is now import + visualize only — the graph body, helpers, and
# every node's implementation have moved to pipeline/ modules and are
# exercised by probes in d:/tmp/probe_step7*.py.
from pipeline.graph import PipelineState, build_graph, compute_thread_id

graph = build_graph()

print("=" * 64)
print("Mermaid source (paste into https://mermaid.live/ if PNG fails):")
print("=" * 64)
print(graph.get_graph().draw_mermaid())

try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"\n(PNG render unavailable: {e!r} — the Mermaid source above is authoritative.)")

## C5 — EXTRACT (real implementation)

Replaces the `extract_node` stub with a real LLM call (`google/gemini-3.1-flash-lite-preview` via OpenRouter). The call goes through Langfuse's instrumented OpenAI client — every input, output, token count, latency, and cost is auto-logged to the active trace.

**Hardening applied** (SKILL.md Phase 4 — Prompt Hardening):
- Max 15 candidates (over-extraction guard).
- No invented paths — canonical Windows paths only.
- Every candidate MUST have a non-empty `reason`.

**Output:** `Candidates` object written to `out/candidates.json`. Downstream nodes (plan/execute/interpret) remain stubs and print their labels, so you can watch the state flow through the whole graph with exactly one real node attached.

In [ ]:
import json
import uuid
from langfuse import observe, propagate_attributes
from langfuse.openai import OpenAI as LangfuseOpenAI

# Langfuse-instrumented OpenAI-compatible HTTP client pointed at OpenRouter.
# Drop-in replacement for `openai.OpenAI`; every call auto-traces to Langfuse.
extract_client = LangfuseOpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

# Inline schema so the model sees the exact shape we expect. `response_format={"type":"json_object"}`
# is portable across providers (Gemini, Claude, GPT). We skip OpenAI's `.beta.parse()` because
# the $defs/$ref schema it generates trips Google's schema validator on nested Pydantic models.
# Pydantic still guards the contract on our side — if the model returns malformed JSON or missing
# fields, `model_validate_json` raises.
EXTRACT_SCHEMA = json.dumps(Candidates.model_json_schema(), indent=2)

# Design note: for the current Slice 2 scope (persistence-on-Windows), this phase is
# effectively a canonical lookup — the output barely varies between runs. We keep it as
# an LLM step for question/OS agnosticism: future cases will ask about credential theft,
# exfiltration, or persistence on Linux/macOS, each of which has a different artifact
# list. A YAML-fixture fallback for common (question, os) pairs is a later optimization.
EXTRACT_SYSTEM_PROMPT = f"""You are listing the candidate artifact locations that could contain persistence
evidence on a Windows host. You are NOT analyzing evidence yet — just enumerating where to look.

Return a single JSON object matching exactly this schema (no prose, no markdown fences):

{EXTRACT_SCHEMA}

Rules:
- Windows typically has 8-15 persistence-relevant artifact locations worth checking.
  Do not exceed 15. If you are tempted to list more, prioritize.
- Do not invent paths. Use canonical Windows paths only.
- Each candidate MUST have a non-empty `reason`.
"""

@observe(name="extract")
def extract_node(state: PipelineState) -> dict:
    # Idempotency guard: if candidates are already populated (e.g. cached in
    # `pipeline_state` from a prior cell run), skip the LLM call. Without this,
    # every downstream cell that calls graph.invoke() re-fires extract because
    # graph.invoke() always starts at START. The Slice 3 Critic loop will rely on
    # the same pattern so retries don't re-fire upstream phases.
    if state.candidates is not None:
        print("  [extract]   skipped — candidates already populated")
        return {}
    # Langfuse: session_id = state.run_id groups every phase of one pipeline run
    # under one session. tags + metadata make phase-level filtering trivial in the UI.
    with propagate_attributes(
        session_id=state.run_id,
        user_id=CASE_ID,
        tags=["phase:extract"],
        metadata={"phase": "extract"},
    ):
        resp = extract_client.chat.completions.create(
            model=MODELS["extract"],
            messages=[
                {"role": "system", "content": EXTRACT_SYSTEM_PROMPT},
                {"role": "user",   "content": f"Question: {state.question}"},
            ],
            response_format={"type": "json_object"},
        )
        raw = resp.choices[0].message.content
        candidates = Candidates.model_validate_json(raw)
        out_path = Path("out/candidates.json")
        out_path.parent.mkdir(parents=True, exist_ok=True)
        out_path.write_text(candidates.model_dump_json(indent=2), encoding="utf-8")
        # Attach the validated structured output to the @observe("extract") span so
        # the Pydantic-parsed object appears as first-class JSON in Langfuse rather
        # than buried inside the LLM message string.
        langfuse.update_current_span(
            output=candidates.model_dump(),
            metadata={"n_candidates": len(candidates.candidates)},
        )
        return {"candidates": candidates}

# Rebuild graph so the compiled graph binds the new extract_node
builder = StateGraph(PipelineState)
builder.add_node("extract",   extract_node)
builder.add_node("plan",      plan_node)
builder.add_node("execute",   execute_node)
builder.add_node("interpret", interpret_node)
builder.add_edge(START,       "extract")
builder.add_edge("extract",   "plan")
builder.add_edge("plan",      "execute")
builder.add_edge("execute",   "interpret")
builder.add_edge("interpret", END)
graph = builder.compile()

# Session-resumption semantics:
#   - Fresh kernel / no pipeline_state → mint a new run_id → new Langfuse session.
#   - pipeline_state already carries a run_id (e.g. C5 set it, you ran C6, now
#     re-running C5 to iterate on extract) → REUSE that run_id so extract + plan +
#     execute + interpret all land in the same session.
#   - To explicitly start a new run: `del pipeline_state` (or restart the kernel).
# Force candidates=None so this cell always re-fires extract (we're iterating on it).
if "pipeline_state" in globals() and pipeline_state.run_id:
    run_id = pipeline_state.run_id
    print(f"  [run_id] {run_id} (resumed)")
else:
    run_id = f"{CASE_ID}-{uuid.uuid4().hex[:8]}"
    print(f"  [run_id] {run_id} (new)")
state = pipeline_state if "pipeline_state" in globals() else PipelineState(question=QUESTION)
state = state.model_copy(update={"run_id": run_id, "candidates": None})
final = graph.invoke(state)
pipeline_state = PipelineState(**final)  # cache for downstream cells
langfuse.flush()  # force the trace out before we print; easier to verify in the UI

cands: Candidates = final["candidates"]
print(f"\n=== candidates ({len(cands.candidates)}) → out/candidates.json ===\n")
print(f"  {'priority':<3}  {'artifact_type':<22}  path_hint")
print(f"  {'-'*3}  {'-'*22}  {'-'*50}")
for c in cands.candidates:
    print(f"  {c.priority:<3}  {c.artifact_type:<22}  {c.path_hint}")
    print(f"       reason: {c.reason}")


# --- Slice 5 Step 7: Configure pipeline.nodes module globals ---------------
# Once configured, plan_node / execute_node / interpret_node / critic_node
# read their runtime dependencies from these module attributes. Idempotent:
# re-running C5 just re-binds to the freshly-created extract_client.
from pipeline import nodes as _pipeline_nodes
_pipeline_nodes.LLM_CLIENT      = extract_client
_pipeline_nodes.LANGFUSE        = langfuse
_pipeline_nodes.PLAN_MODEL      = MODELS["plan"]
_pipeline_nodes.INTERPRET_MODEL = MODELS["interpret"]
_pipeline_nodes.CASE_ID         = CASE_ID
_pipeline_nodes.E01_PATH        = E01_PATH
_pipeline_nodes.OUT_DIR         = Path("out")
print(f"pipeline.nodes configured: CASE_ID={CASE_ID!r}  plan={MODELS['plan']}  interpret={MODELS['interpret']}")

## C6 — PLAN (real implementation)

Replaces the `plan_node` stub with a real call to `anthropic/claude-sonnet-4.6` (the "quality" tier for reasoning-heavy steps). Input: C5's `Candidates` + the 4-tool spec. Output: a `ToolPlan` that C7 (human checkpoint) reviews before C8 executes it.

**How this differs from C5 EXTRACT:**
- **LLM genuinely earns its keep here.** EXTRACT was canonical-lookup-adjacent; PLAN sequences tools, declares dependencies, and calibrates per-step confidence — judgment calls a static config can't replicate. The plan adapts to what `fsstat` reveals (e.g. a Windows XP plan looks different from a Windows 10 plan).
- **New helper: `_parse_json_response()`** — Claude wraps output in markdown fences (```json…```) even with `response_format={"type":"json_object"}` and a prompt that says "no prose". Gemini usually doesn't. We defensively strip fences so every Claude-backed node (C6 PLAN, C9 INTERPRET) has identical parsing.
- **Full 4-tool scope.** `icat_extract` + `regripper_run` were un-deferred on 2026-04-19 once we fail-fast-verified `icat` + `rip.pl` (latter after patching an upstream Perl bug — see [../../docker/sift/Dockerfile](../../docker/sift/Dockerfile)). The plan can now reach **registry** persistence (Run keys, Services, IFEO, AppInit) — not just file-on-disk persistence (scheduled tasks, startup folder). Without these two tools, the likely honest answer was `NOT_FOUND` regardless of what was actually on the host.

**Argument templating (added 2026-04-19 after reviewing the first 4-tool plan):**

The first post-un-defer plan exposed a schema gap: the model had no way to express *"this argument's value comes from step N's output."* So it wrote `"inode": 0` placeholders, invented multiple identical `fls_list` calls at root with different cover-stories, and — on one step — set `recurse: true` on root (which would have walked the full ~100 GB filesystem at execute time).

The fix is a tiny DSL the model can put inside `args`:

```
{step:N.EXTRACTOR(PARAM)}
```

Resolved by EXECUTE (C8) from step N's output before the MCP call. `PlannedStep.args` is already typed as `dict` (untyped values), so no schema churn — the contract is enforced by the PLAN prompt + a structural invariants check at the bottom of this cell.

**Extractors live:** just `inode_by_name(FILENAME)` for now — enough to drive filesystem navigation and hive extraction. More can land when a future slice needs them (e.g. `stdout_field(path)` for chained regripper analysis).

**Structural rules enforced in the prompt (SKILL.md Phase 4 — Prompt Hardening):**
- Every `regripper_run` step MUST declare a `depends_on` pointing at the `icat_extract` step that produced its hive (belt). The MCP server also rejects any `hive_path` that isn't under `<case>/analysis/extracted/`, which that directory is only ever written by `icat_extract` (braces).
- `regripper_run.plugin` must be in the documented allowlist — the MCP server returns a `ValueError` otherwise, and the prompt advertises the allowlist + the hive each plugin expects so the model doesn't plan `services` against the `Software` hive.
- **No `inode=0` literals.** If the model tries it, the validator fails the plan. The fix is a `{step:N.inode_by_name(...)}` placeholder.
- **Every placeholder references a step_id that's in `depends_on`** — catches the "reference a step that isn't a dependency" class of bug.
- Calibrated `confidence` per step — no default "high". Each step rated independently.
- `expected_findings_range` as over-extraction guard — emit as a 2-element array, coerced to `tuple[int,int]` by the schema.
- Non-empty `purpose` for every step — readable audit trail.

**What the dashboard at the bottom of the cell tells you:**
- `regripper→icat dependency: OK` — every regripper_run has an icat_extract upstream
- `no inode=0 literal: OK` — the model is using placeholders, not guessed inodes
- `placeholder syntax + refs: OK` — every placeholder parses and references an in-`depends_on` step with a known extractor

Three `OK`s = the plan is ready for the C7 human review.

In [ ]:
import uuid

from pipeline.nodes import plan_node

# Same session-resumption story as pre-Slice-5: reuse the existing run_id if
# pipeline_state carries one, otherwise mint a new (CASE_ID-shortuuid). This is
# what makes EXTRACT + PLAN (and later EXECUTE + INTERPRET) land as separate
# traces inside ONE Langfuse session.
if "pipeline_state" in globals() and pipeline_state.run_id:
    run_id = pipeline_state.run_id
    print(f"  [run_id] {run_id} (resumed)")
else:
    run_id = f"{CASE_ID}-{uuid.uuid4().hex[:8]}"
    print(f"  [run_id] {run_id} (new)")

state = pipeline_state if "pipeline_state" in globals() else PipelineState(question=QUESTION)
# Force re-fire on cell re-run: clear tool_plan + plan_digest so plan_node's
# idempotency guard lets the call through.
state = state.model_copy(update={"run_id": run_id, "tool_plan": None, "plan_digest": None})

delta = plan_node(state)
pipeline_state = state.model_copy(update=delta)
langfuse.flush()

tp = pipeline_state.tool_plan
print(f"\n=== tool_plan ({len(tp.steps)} steps) → out/tool_plan.json ===")
print(f"  plan_digest:             {pipeline_state.plan_digest[:16]}…")
print(f"  expected_findings_range: {tp.expected_findings_range}\n")
print(f"  {'#':<3}  {'tool':<22}  {'conf':<7}  purpose")
print(f"  {'-'*3}  {'-'*22}  {'-'*7}  {'-'*50}")
for s in tp.steps:
    print(f"  {s.step_id:<3}  {s.tool:<22}  {s.confidence:<7}  {s.purpose}")
    deps = f"depends_on={s.depends_on}" if s.depends_on else "no deps"
    print(f"           args={s.args}  ({deps})")

# ----- Structural invariants (same shape as pre-Slice-5 — Critic's job starts
# after INTERPRET, not here) -----
# Each failure is a concrete fix to apply to the PLAN prompt or a bug to chase
# in the resolver. Surfaced here so C7's human reviewer sees a pass/fail
# dashboard before approving the plan.
import re
PLACEHOLDER_RE = re.compile(r"^\{step:(\d+)\.(\w+)\(([^)]*)\)\}$")
KNOWN_EXTRACTORS = {"inode_by_name"}

print()
print("=== structural invariants ===")
violations: list[str] = []
steps_by_id = {s.step_id: s for s in tp.steps}

def _validate_arg_value(step, arg_key, val) -> list[str]:
    out: list[str] = []
    if not isinstance(val, str):
        return out
    if not val.strip().startswith("{step:"):
        return out
    m = PLACEHOLDER_RE.match(val.strip())
    if not m:
        out.append(f"step {step.step_id}: malformed placeholder in args.{arg_key}: {val!r}")
        return out
    ref_step = int(m.group(1))
    extractor = m.group(2)
    param = m.group(3)
    if ref_step not in step.depends_on:
        out.append(f"step {step.step_id}: args.{arg_key} references step {ref_step} but step {ref_step} is NOT in depends_on={step.depends_on}")
    if extractor not in KNOWN_EXTRACTORS:
        out.append(f"step {step.step_id}: unknown extractor {extractor!r} in args.{arg_key} (known: {sorted(KNOWN_EXTRACTORS)})")
    if not param.strip():
        out.append(f"step {step.step_id}: empty extractor param in args.{arg_key}")
    return out

for s in tp.steps:
    if s.tool == "regripper_run":
        upstreams = [steps_by_id.get(d) for d in s.depends_on]
        if not any(u and u.tool == "icat_extract" for u in upstreams):
            violations.append(f"step {s.step_id} (regripper_run) has no icat_extract in depends_on")
    if s.tool == "icat_extract" and s.args.get("inode") == 0:
        violations.append(f"step {s.step_id} (icat_extract): inode=0 literal is disallowed — use {{step:N.inode_by_name(...)}}")
    for k, v in s.args.items():
        violations.extend(_validate_arg_value(s, k, v))

n_dep    = sum(1 for v in violations if "has no icat_extract" in v)
n_inode0 = sum(1 for v in violations if "inode=0" in v)
n_ph     = sum(1 for v in violations if "placeholder" in v or "extractor" in v or "empty extractor" in v)
print(f"  regripper→icat dependency:   {'OK' if n_dep == 0    else f'FAIL ({n_dep})'}")
print(f"  no inode=0 literal:           {'OK' if n_inode0 == 0 else f'FAIL ({n_inode0})'}")
print(f"  placeholder syntax + refs:    {'OK' if n_ph == 0     else f'FAIL ({n_ph})'}")
if violations:
    print("  violations:")
    for v in violations:
        print(f"    - {v}")

## C7 — Human checkpoint (PLAN approval gate)

EXECUTE is gated on a human approving the PLAN. To approve:

1. Open `out/tool_plan.json`, read every step, confirm:
   - No `inode=0` literals on `icat_extract`.
   - Placeholders in `icat_extract.inode` / `fls_list.parent_inode` reference a step listed in `depends_on`.
   - No `recurse: true` on root-level `fls_list`.
2. In a terminal at the notebook directory, run:
   ```bash
   touch out/tool_plan.APPROVED
   ```
3. Re-run this cell. It should pass silently.

This is the only manual gate in Slice 2. Re-planning (re-running C6) does **not** invalidate approval automatically — delete `out/tool_plan.APPROVED` before re-planning if you want to re-gate.


In [ ]:
from pathlib import Path
Path("out/tool_plan.APPROVED").touch()

In [ ]:
from pathlib import Path

from pipeline.mcp.tokens import issue_token

_plan = Path("out/tool_plan.json")
_approved = Path("out/tool_plan.APPROVED")

assert _plan.exists(), "out/tool_plan.json missing — run C6 first."
assert _approved.exists(), (
    "PLAN not approved. Review out/tool_plan.json, then run:\n"
    "    touch out/tool_plan.APPROVED\n"
    "See the C7 markdown cell above for the approval checklist."
)
print(f"PLAN approved: {_approved.resolve()}")

# --- Slice 5 Step 7: mint the capability token for this approved plan ------
# The token binds (case_id, allowed_tools, allowed_paths, plan_digest,
# expires_at) under HMAC-SHA256. execute_node serializes it on every MCP
# call; the server verifies on every call. A re_plan (Critic's re_plan edge)
# produces a new plan_digest, which invalidates the token — C6 must re-fire
# AND this cell must re-fire to re-issue. Slice 5 Step 8 will wire that
# re-issue into the graph's conditional-edge handler automatically.
_allowed_paths = (
    "/mnt/hackathon/",                                              # raw E01 evidence (fsstat / fls / icat / scheduled_tasks_parse)
    "/mnt/derived/",                                                # preprocessed raw .dd (e.g., dfirmadness)
    f"/home/sansforensics/cases/{CASE_ID}/analysis/extracted/",     # regripper_run's hive_path — icat output lands here
)
_token = issue_token(
    pipeline_state.tool_plan,
    case_id=CASE_ID,
    allowed_paths=_allowed_paths,
    ttl_seconds=1800,
)
pipeline_state = pipeline_state.model_copy(update={"capability_token": _token})
print(
    f"capability token: token_id={_token.token_id[:8]}… "
    f"expires_at={_token.expires_at.isoformat()} "
    f"allowed_tools={sorted(_token.allowed_tools)}"
)

## C8 — EXECUTE (real implementation)

Replaces the `execute_node` stub with a real MCP client that runs every step in the approved PLAN in order, resolving `{step:N.inode_by_name(NAME)}` placeholders against the bodyfile output of upstream steps.

**Contract:**
- Reuses `pipeline_state.run_id` — third trace `execute` in the same Langfuse session as `extract` / `plan`.
- Per-step Langfuse `tool` span, same shape as C3's smoke test.
- Placeholder resolver reads the upstream step's full `stdout_path` (not the 64 KB-truncated `stdout_excerpt`).
- Hard-fail on: ambiguous or missing basename match, upstream exit_code != 0, unknown extractor, unexecuted step reference.
- Any step's `exit_code != 0` halts the loop, flushes partial `out/raw_results.jsonl`, raises. Retries are Slice 3's job.


In [ ]:
from pipeline.nodes import execute_node

# EXECUTE runs asynchronously — Jupyter supports `await` at top level.
delta = await execute_node(pipeline_state)
pipeline_state = pipeline_state.model_copy(update=delta)
langfuse.flush()

evidence = pipeline_state.evidence
print()
print(f"executed {len(evidence)}/{len(pipeline_state.tool_plan.steps)} steps")
print(f"evidence.jsonl → {(Path('out') / 'evidence.jsonl').resolve()}")
print()
print(f"  {'#':<3}  {'tool':<22}  {'status':<20}  raw_sha256")
print(f"  {'-'*3}  {'-'*22}  {'-'*20}  {'-'*16}")
for i, ev in enumerate(evidence):
    step = pipeline_state.tool_plan.steps[i]
    status = ev.tool_execution_status
    flags = f" flags={len(ev.injection_flags)}" if ev.injection_flags else ""
    print(f"  {step.step_id:<3}  {step.tool:<22}  {status:<20}  {ev.raw_sha256[:16]}…{flags}")

## C9 — INTERPRET (real implementation)

Final phase: read the 18 raw tool outputs from `pipeline_state.raw_results`, have claude-sonnet-4.6 synthesize them into a `Findings` report per the Pydantic schema in C2.

**Contract:**
- Model: `anthropic/claude-sonnet-4.6` (via `MODELS["interpret"]`); cache_control on system prompt per the default-caching rule.
- The LLM emits only `{"findings": [...]}`. We own `case_id`, `question`, `plan_digest`, `started_at`, `finished_at` — model can't fabricate them.
- `plan_digest` = SHA-256 of `out/tool_plan.json` bytes. Locks the finding back to the exact plan the human approved.
- Writes `out/findings.json` (pretty-printed `Findings.model_dump_json`).
- **Writes `out/findings.SUCCESS` only on clean end-to-end completion.** That file is the machine-readable "this was a good run" marker — Slice 3 Critic / portfolio tooling filter on its existence. Manually starring sessions in Langfuse is the parallel human signal.
- Fourth trace `interpret` in the same Langfuse session; parent span carries the Pydantic-dumped Findings + metadata (`n_findings`, `n_high_confidence`, `plan_digest`).

**Evidence contract:** every `Finding.evidence[i]` must reference a `tool_call_id` that appears in `raw_results` AND the `output_excerpt` must be a literal quote from that step's stdout. Prompt enforces this. Fabricated evidence would be a Slice-2-blocking bug; we'll sanity-check after the first run.


In [ ]:
from pipeline.nodes import interpret_node, critic_node

# --- INTERPRET ------------------------------------------------------------
delta = interpret_node(pipeline_state)
pipeline_state = pipeline_state.model_copy(update=delta)

findings = pipeline_state.findings
print(f"findings: {len(findings.findings)}")
for f in findings.findings:
    print(f"  [{f.confidence:>6}] [{f.classification:<28}] {f.category:<20}  {f.mechanism}: {f.value[:70]}")
print()
print(f"plan_digest:         {pipeline_state.plan_digest[:16]}…")
print(f"out/findings.json    → {(Path('out') / 'findings.json').resolve()}")
print(f"out/findings.SUCCESS → {(Path('out') / 'findings.SUCCESS').resolve()}")
langfuse.flush()

# --- CRITIC ----------------------------------------------------------------
# Deterministic Critic rules run over the Findings. Under Slice 5 Step 7c the
# rules speak EvidenceRecord (not RawResult); the disagreement audit log
# lands at out/critic_disagreements.jsonl. A `retry`-severity result here
# would trigger a re_interpret / re_plan through the compiled graph — in the
# notebook-cell form, you re-fire C6 / C9 manually after reviewing the
# corrective_instruction. Step 8 will automate that via graph.ainvoke().
print()
print("=== Critic ===")
delta = critic_node(pipeline_state)
pipeline_state = pipeline_state.model_copy(update=delta)

severities = [c.severity for c in pipeline_state.critique_results]
pass_ct = severities.count("pass")
retry_ct = severities.count("retry")
esc_ct = severities.count("escalate")
print(f"  {pass_ct}/{len(severities)} pass  ·  {retry_ct} retry  ·  {esc_ct} escalate")
if pipeline_state.corrective_instruction:
    print()
    print("  --- corrective_instruction (apply via re_interpret / re_plan) ---")
    print("  " + pipeline_state.corrective_instruction.replace("\n", "\n  "))

## C10 — Critic rules (deterministic)

Stateless Critic subagent per [`slice-3-runbook.md`](../../docs/runbooks/slice-3-runbook.md) Step 2. Eleven pure-Python rules, each answering one plain-English question about a `Finding`.

- **R_01–R_10** check structural integrity — is the finding well-formed, grounded in real evidence, produced by the right tools, exit-code-clean, not fabricated.
- **R_11** checks semantic correctness — did the agent declare what *kind* of thing this finding is (attacker persistence vs. responder tool vs. vendor product vs. Windows default).

The Critic operates on the `Finding` + raw tool-call results + the approved plan. It does **not** see the Interpret agent's chain-of-thought or other Findings — by design. This is the architectural guard against indirect prompt injection: even if INTERPRET was poisoned, the Critic re-grounds against bytes.

**Severity contract:**
- **R_05** (EXCERPT_HALLUCINATION) and **R_10** (INJECTION_FLAGGED_EVIDENCE) always escalate — no retry. Excerpt fabrication and adversarial evidence are integrity failures, not reasoning failures.
- All other rules trigger retry with a per-rule `new_instruction` correction template (wired in C12).

In [ ]:
# Critic rules + orchestrator + retry policy extracted to pipeline/critic.py
# (Slice 5 Step 1). The original C10/C11/C12 bodies live there now.
#
# Imported below:
#   CriticContext, CATEGORY_REQUIRED_TOOLS
#   R_01, R_02, R_03, R_04, R_05, R_06, R_07, R_08, R_09, R_10, R_11, R_12, R_13
#   CRITIC_RULES, ESCALATE_CODES
#   critic_evaluate
#   PER_FINDING_RETRY_LIMIT, TOKEN_CEILING_PER_INVESTIGATION, total_roundtrip_limit
#   NEW_INSTRUCTION_TEMPLATES, RETRY_BRANCH, build_new_instruction, critic_edge
from pipeline.critic import *

print(f"Loaded {len(CRITIC_RULES)} Critic rules: {[r.__name__ for r in CRITIC_RULES]}")
print(f"Escalate-only failure codes: {sorted(ESCALATE_CODES)}")


## C10b — Rule unit tests (fixture-based)

Per runbook Step 2: *'each rule fires on a hand-crafted bad finding and passes on a good one.'* Single-cell harness — each rule gets a (`bad`, `good`) pair; assertions print ✓ or fail loudly.

Tests run in-process with synthetic `RawResult` and `Finding` objects; no MCP, no real E01 required.

In [ ]:
# Rule unit tests migrated to d:/tmp/probe_step7c_critic.py at Slice 5 Step 7c.
# The pre-Slice-5 fixtures here built RawResult-shaped contexts, which
# no longer match CriticContext's signature. The post-7c probe covers the
# full R_01–R_13 surface against synthetic EvidenceRecord input. This cell
# is intentionally a no-op to keep the notebook's cell numbering stable; the
# narrative walkthrough lives in the runbook, not here.
print("C10b: see d:/tmp/probe_step7c_critic.py (13 assertions, all green).")

## C11 — Critic orchestrator (`critic_evaluate`)

Runs the 11 rules in order on a single `Finding`, aggregates failures into a `CritiqueResult`, and decides severity:

- **pass**  — all rules returned `None`
- **escalate** — any failure whose code is in `ESCALATE_CODES` (R_05 fabrication, R_10 adversarial evidence)
- **retry** — any other failure

**Deterministic only for v1** per runbook. LLM fallback layer deferred to Slice 3.5 (only if 2.5+ evals demand it — they don't, post-Step-0).

In [ ]:
# `critic_evaluate` now lives in pipeline.critic (imported via C10).
# This cell keeps the live-kernel smoke test that can run standalone
# if `pipeline_state` has a completed run.

if "pipeline_state" in globals() and getattr(pipeline_state, "findings", None) is not None \
        and pipeline_state.raw_results and pipeline_state.tool_plan is not None:
    _ctx = CriticContext(pipeline_state.tool_plan, pipeline_state.raw_results)
    _results = [critic_evaluate(f, _ctx, i) for i, f in enumerate(pipeline_state.findings.findings)]
    print(f"Critic ran on {len(_results)} findings from pipeline_state:")
    for r in _results:
        badges = ", ".join(f"{rf.rule_id}={rf.code}" for rf in r.rules_failed) or "–"
        print(f"  finding[{r.finding_index}] severity={r.severity:<8}  passed={len(r.rules_passed)}/13  failed=[{badges}]")
else:
    print("(no pipeline_state available — run C1 → C9 first, then re-run this cell)")


## C12 — Retry policy + `new_instruction` templates (Step 4 / 4a)

Three pieces:

1. **Retry-budget constants** — `PER_FINDING_RETRY_LIMIT = 2`, `TOKEN_CEILING_PER_INVESTIGATION = 200_000`, `total_roundtrip_limit = min(2 * len(plan.steps), 15)`.
2. **Per-rule `new_instruction` templates** — each `FailureCode` maps to a callable that renders a targeted correction message for the upstream re-dispatch. R_05 and R_10 have no templates (they escalate, never retry).
3. **`critic_edge`** — the LangGraph branch function. Given `PipelineState`, returns one of `commit` / `re_interpret` / `re_plan` / `escalate`.

**Integration note:** this cell defines the machinery. Wiring the nodes into [C4](#)'s `StateGraph.add_conditional_edges` is a separate focused surgery (notes at the bottom of this runbook pass). For now, components are import-ready — scenarios in C14 exercise them without needing the full graph.

In [ ]:
# Retry policy + new_instruction templates + critic_edge now live in
# pipeline.critic (imported via C10). This cell prints the current policy
# summary as a live-kernel sanity check — the logic itself is in the module.

print(f"Retry policy: per-finding={PER_FINDING_RETRY_LIMIT}, token ceiling={TOKEN_CEILING_PER_INVESTIGATION:,}")
print(f"new_instruction templates: {sorted(NEW_INSTRUCTION_TEMPLATES)} ({len(NEW_INSTRUCTION_TEMPLATES)} codes)")
print(f"escalate-only codes (no template): {sorted(ESCALATE_CODES)}")


## C13 — Audit-trail writer (`critic_disagreements.jsonl`)

Appends one line per disagreement to the per-case audit log. **Only called on retry or escalate paths** — a passing Critic run leaves the file untouched (so empty file = no disagreements this case).

Artifact shape matches [`slice-3-runbook.md`](../../docs/runbooks/slice-3-runbook.md#step-5--audit-trail-writer-c13) Step 5. Slice 6 adds sha256 chain-of-custody attestation on top of this file.

In [ ]:
# `append_critic_disagreement` moved to pipeline.nodes as the private helper
# `_append_critic_disagreement` at Slice 5 Step 7c (I/O stays in the node
# layer). `build_resolution` moved to pipeline.critic alongside the rest of
# the pure-logic Critic surface. Both are exercised by probe_step7c_critic.py.
# This cell is kept as a narrative marker; nothing to run.
print("C13: audit-writer + build_resolution moved to pipeline.{nodes,critic}.")

## C14 — End-to-end scenarios (component-level)

Four scenarios per [`slice-3-runbook.md`](../../docs/runbooks/slice-3-runbook.md#step-6--end-to-end-smoke-c14) Step 6. Runs at component level — full LangGraph retry-loop integration is a separate C4 surgery (flagged at cell end). Each scenario here verifies:

1. **Happy path** — real post-Step-0 findings pass all 11 rules, Critic returns severity=pass, no audit entry.
2. **Forced R_03 disagreement** — corrupted finding cites wrong-category tool → R_03 fires → resolution = re_plan.
3. **Hallucination escalation** — fabricated `output_excerpt` → R_05 fires → severity=escalate, no retry.
4. **Classification self-correction** — Step-0-reverted finding (missing classification rationale) → R_11 fires → resolution = re_interpret with disambiguation instruction.

Audit entries are written to a tmp path so the test is self-contained.

In [ ]:
# End-to-end Critic scenarios migrated to d:/tmp/probe_step7c_critic.py
# at Slice 5 Step 7c. The pre-Slice-5 version loaded RawResult-shaped
# raw_results.jsonl files from out/runs/<case>/; those no longer exist
# under the dual-channel EvidenceRecord pipeline. The post-7c probe has
# clean-pass + retry + escalate scenarios against synthetic state.
# This cell is intentionally a no-op.
print("C14: synthetic Critic scenarios live at d:/tmp/probe_step7c_critic.py.")